# **Домашнее задание 1: Оптический поток и динамика видео**

### **Цель**

Освоить практическое применение оптического потока для анализа движения, научиться строить полные видеопайплайны, включающие вычисление потока, фильтрацию, warping и стабилизацию. Студент выбирает **одну** из двух инженерных задач: построение мини-системы **стабилизации камеры** или создание **трекинг-модуля движения**, анализирующего траектории и устойчивость методов Lucas–Kanade и Farnebäck.

---

# **Вариант A: Мини-система стабилизации камеры**

### **Задание**

1. Выбрать видеоролик с заметной дрожью камеры (телефонная съёмка, action-камера, GoPro, записанная вручную панорама).
2. Вычислить плотный оптический поток (Farnebäck) или разреженный (LK) и оценить глобальное движение камеры по кадрам (аффинная модель или гомография).
3. Сгладить траекторию движения низкочастотным фильтром (скользящее окно, экспоненциальное сглаживание или фильтр Калмана по желанию).
4. Выполнить **компенсацию движения** через warp каждого кадра к стабилизированной траектории.
5. Сформировать стабилизированное видео.
6. Построить визуальные сравнения «до/после» и провести **error analysis**: какие участки стабилизируются плохо и почему (оптический поток, тени, motion blur, нехватка текстуры, ошибки глобальной модели).


---

# **Требования к сдаче**

Репозиторий на GitHub:

* `src/` или ноутбук с кодом вычисления оптического потока и всего пайплайна.
* `README.md` c описанием выбранного варианта (A или B), параметров, применённых фильтров и основных наблюдений.
* Визуализации:
  – для варианта A: примерные кадры «до/после», графики траектории движения камеры, примеры warping-а;
  – для варианта B: траектории LK, карты плотного потока, маски движения.
* При варианте A — итоговое стабилизированное видео.
* При варианте B — сравнительный отчёт о различиях LK и Farnebäck.

---

# **Критерии оценки**

| Баллы | Критерий                                                                      |
| ----- | ----------------------------------------------------------------------------- |
| 0–3   | Полнота: реализованы все этапы выбранного варианта (A или B).                 |
| 0–3   | Код: чистый, воспроизводимый, корректно использует OpenCV.                    |
| 0–2   | Анализ: качественный разбор ошибок, интерпретация результатов, выводы.        |
| 0–2   | Репозиторий: аккуратность, читаемость, примеры визуализаций, понятный README. |

**Максимум: 10 баллов.**

---


# Вариант A

In [3]:
import cv2
from google.colab.patches import cv2_imshow
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt


SMOOTHING_RADIUS = 30

In [4]:
# реализация методов вычисления оптического потока

# LK - разреженный оптический поток
def calculate_KL(prev_gray, curr_gray):
  # обнаружение углов на кадре
  prev_points = cv2.goodFeaturesToTrack(
      prev_gray, # предыдущий кадр
      maxCorners=200,
      qualityLevel=0.01,
      minDistance=5
  )

  # Если точек мало или нет, возвращаем единичную матрицу (движения нет)
  if prev_points is None:
      return np.eye(2, 3)

  curr_points, status, err = cv2.calcOpticalFlowPyrLK(prev_gray,
          curr_gray,
          prev_points,
          None)

  # оставляем только статус = 1 (точки, которые успешно отследились)
  idx = np.where(status == 1)[0]
  # оставляем только валидные точки для предыдущего и текущего кадров
  prev_pts_valid = prev_points[idx]
  curr_pts_valid = curr_points[idx]

  # нужно минимум 2 точки для estimateAffinePartial2D
  if len(prev_pts_valid) < 2:
      return np.eye(2, 3)
  # афинное преобразование
  m, _ = cv2.estimateAffinePartial2D(prev_pts_valid, curr_pts_valid)

  if m is None:
      return np.eye(2, 3)

  return m

# Farneback - плотный оптический поток
def calculate_Farneback(prev_gray, curr_gray):
  # размеры кадра
  h, w = prev_gray.shape[:2]

  flow = cv2.calcOpticalFlowFarneback(
    prev_gray, curr_gray, None,
    pyr_scale=0.5,
    levels=5,
    winsize=15,
    iterations=5,
    poly_n=5,
    poly_sigma=1.2,
    flags=0
    )

  # Берем сетку с шагом 16 пикселей, чтобы было быстрее
  step = 16
  y, x = np.mgrid[step/2:h:step, step/2:w:step].reshape(2,-1).astype(int)

  # векторы движения этих точек из flow
  fx = flow[y, x, 0]
  fy = flow[y, x, 1]

  # сравниваем как было и как стало
  pts_prev = np.vstack([x, y]).T.reshape(-1, 1, 2).astype(np.float32)
  pts_curr = np.vstack([x + fx, y + fy]).T.reshape(-1, 1, 2).astype(np.float32)

  # афинное преобразование
  m, _ = cv2.estimateAffinePartial2D(pts_prev, pts_curr)

  if m is None:
      return np.eye(2, 3)

  return m

In [5]:
# вспомогательные функции

def moving_average(curve, radius):
    window_size = 2 * radius + 1
    # ядро фильтра
    f = np.ones(window_size) / window_size
    # padding, чтобы длина массива не изменилась
    curve_pad = np.pad(curve, (radius, radius), 'edge')
    # свертка
    curve_smoothed = np.convolve(curve_pad, f, mode='same')
    # убираем padding, возвращаем исходный размер
    return curve_smoothed[radius:-radius]


def smooth_trajectory(trajectory, radius):
    # Сглаживает траекторию по осям X, Y, Угол
    smoothed = np.copy(trajectory)
    # 0 - x, 1 - y, 2 - angle
    for i in range(3):
        smoothed[:, i] = moving_average(trajectory[:, i], radius)
    return smoothed


def plot_trajectory_graph(trajectory,smoothed_trajectory, METHOD,output_plot_path):

  plt.figure(figsize=(10, 5))
  plt.plot(trajectory[:, 0], label='Original X')
  plt.plot(smoothed_trajectory[:, 0], label='Smoothed X', linewidth=2)
  plt.plot(trajectory[:, 1], label='Original Y')
  plt.plot(smoothed_trajectory[:, 1], label='Smoothed Y', linewidth=2)
  plt.title(f'Camera Trajectory ({METHOD})')
  plt.legend()
  plt.grid(True)
  plt.savefig(output_plot_path) # Сохраняем график
  plt.close()

In [6]:
def video_processing(input_video_path, output_video_path, METHOD):

  # видео сравнение
  output_video_path_diff = output_video_path.replace('.mp4', '_diff.mp4')

  cap = cv2.VideoCapture(input_video_path)
  if not cap.isOpened():
      print("Не удалось открыть видео")
      return

  # метаданные видео
  n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # кол-во кадров
  w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))   # ширина кадра
  h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))  # высота кадра
  fps = cap.get(cv2.CAP_PROP_FPS)  # частота

  print(f"---> Processing video: {n_frames} frames, {w}x{h}, {fps} FPS using {METHOD}")

  # Видеорайтер для сохранения
  fourcc = cv2.VideoWriter_fourcc(*'mp4v')
  # стабилизированное видео
  out_stab = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))
  # видео сравнения
  out_diff = cv2.VideoWriter(output_video_path_diff, fourcc, fps, (2 * w, h))

  ret, prev = cap.read() # первый кадр
  # prev - матрица размера h x w x c=3 (rgb)
  # из ргб в чб
  prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)  # h x w x c=1

  # Массив для сохранения смещений между кадрами [dx, dy, da]
  # n_frames - 1 трансформаций
  transforms = np.zeros((n_frames - 1, 3), np.float32)

  # вычисляем оптический поток
  print("Step 1: Calculating optical flow...")
  for frame_idx in tqdm(range(n_frames - 1)):
    # читаем следующий кадр
    success, curr = cap.read()
    if not success:
      break
    # из ргб в чб
    curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)  # h x w x c=1
    # вычисляем оптический поток покадрово разными методами
    if METHOD == 'LK':
      m = calculate_KL(prev_gray, curr_gray)
    elif METHOD == 'Farneback':
      m = calculate_Farneback(prev_gray, curr_gray)


    dx = m[0, 2]
    dy = m[1, 2]
    # угол поворота из матрицы (da = arctan(sin, cos))
    da = np.arctan2(m[1, 0], m[0, 0])

    transforms[frame_idx] = [dx, dy, da]

    # перезаписываем для следующего кадра
    prev_gray = curr_gray

  # построение траектории
  # абсолютная траекторию (кумулятивная сумма смещений)
  trajectory = np.cumsum(transforms, axis=0)

  # сглаживаем траекторию (скользящим средним)
  smoothed_trajectory = smooth_trajectory(trajectory, SMOOTHING_RADIUS)

  # разница между сглаженным путем и реальным (ошибка, которую нужно компенсировать)
  difference = smoothed_trajectory - trajectory
  transforms_smooth = difference

  # графики
  plot_trajectory_graph(trajectory,smoothed_trajectory, METHOD,output_plot_path)

  print("Step 2: Applying stabilization...")
  # Сбрасываем видеопоток на начальные кадр
  cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
  # Читаем первый кадр снова, чтобы начать цикл записи
  cap.read()

  for i in tqdm(range(n_frames - 2)):
        success, frame = cap.read()
        if not success:
            break

        # поправки для текущего кадра
        dx = transforms_smooth[i, 0]
        dy = transforms_smooth[i, 1]
        da = transforms_smooth[i, 2]

        # матрица трансформации из (dx, dy, da)
        # сдвигает кадр на разницу между дрожащей и плавной траекторией
        m = np.zeros((2, 3), np.float32)
        m[0, 0] = np.cos(da)
        m[0, 1] = -np.sin(da)
        m[1, 0] = np.sin(da)
        m[1, 1] = np.cos(da)
        m[0, 2] = dx
        m[1, 2] = dy

        # аффинное преобразование к кадру
        frame_stabilized = cv2.warpAffine(frame, m, (w, h), borderMode=cv2.BORDER_REPLICATE)

        # сохраняем стабилизированное видео
        out_stab.write(frame_stabilized)

        # объединяем кадры для сравнения (Original | Stabilized)
        combined = np.concatenate((frame, frame_stabilized), axis=1)

        # Добавляем текст на кадр
        cv2.putText(combined, "Original", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.putText(combined, f"Stabilized ({METHOD})", (w + 10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        out_diff.write(combined)


  cap.release()
  out_stab.release()
  out_diff.release()

  cv2.destroyAllWindows()

  print(f"Stabilized video saved to: {output_video_path}")
  print(f"Comparison video saved to: {output_video_path_diff}")
  print(f"Plot saved to {output_plot_path}")


In [7]:

# Farneback - плотный оптический поток
# LK - разреженный оптический поток
METHOD = 'Farneback'

# input_path = 'video_data/cv_HW1.mp4'
input_video_path = "/content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/CV_hw_1.mp4"
output_video_path = f"/content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/CV_hw_1_stabilized_result_{METHOD}.mp4"
output_plot_path = f"/content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/trajectory_plot_{METHOD}.png" # Путь для графика

video_processing(input_video_path, output_video_path, METHOD)

---> Processing video: 412 frames, 1920x1080, 30.0 FPS using Farneback
Step 1: Calculating optical flow...


100%|██████████| 411/411 [08:20<00:00,  1.22s/it]


Step 2: Applying stabilization...


100%|██████████| 410/410 [00:41<00:00,  9.80it/s]

Stabilized video saved to: /content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/CV_hw_1_stabilized_result_Farneback.mp4
Comparison video saved to: /content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/CV_hw_1_stabilized_result_Farneback_diff.mp4
Plot saved to /content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/trajectory_plot_Farneback.png


In [8]:

# Farneback - плотный оптический поток
# LK - разреженный оптический поток
METHOD = 'LK'

# input_path = 'video_data/cv_HW1.mp4'
input_video_path = "/content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/CV_hw_1.mp4"
output_video_path = f"/content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/CV_hw_1_stabilized_result_{METHOD}.mp4"
output_plot_path = f"/content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/trajectory_plot_{METHOD}.png" # Путь для графика

video_processing(input_video_path, output_video_path, METHOD)

---> Processing video: 412 frames, 1920x1080, 30.0 FPS using LK
Step 1: Calculating optical flow...


100%|██████████| 411/411 [00:35<00:00, 11.74it/s]


Step 2: Applying stabilization...


100%|██████████| 410/410 [00:42<00:00,  9.64it/s]

Stabilized video saved to: /content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/CV_hw_1_stabilized_result_LK.mp4
Comparison video saved to: /content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/CV_hw_1_stabilized_result_LK_diff.mp4
Plot saved to /content/drive/MyDrive/ИПИИ'25 вышка/CV/video_data/trajectory_plot_LK.png
